In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-deepseek

In [28]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:
llm = ChatDeepSeek(
    model="deepseek-chat", 
    temperature=0.4, 
    api_key="sk-b03ebd023fa744daa62e8d2c0d899111" 
)
# Initialize an empty conversation history.
chat_history = []

Student_Question_Prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an Elementary School Teaching Assistant. "
        "Explain things simply for kids in grades 1-5. "
        "Use analogies, provide numbered steps, and a final answer. "
        "Always suggest 3 similar practice questions at the end."
    )),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# 4. THE FUNCTION
def ask_teacher_question(student_question, history_list):
    # Prepare the chain
    chain = Student_Question_Prompt | llm | StrOutputParser()
    
    # Run the chain passing the history list
    response = chain.invoke({
        "question": student_question,
        "history": history_list
    })
    
    # Update the history list (Assignment 1 logic)
    history_list.append(HumanMessage(content=student_question))
    history_list.append(AIMessage(content=response))
    
    return response

# --- TEST IT ---
# Turn 1
print("--- TEST 1 ---")
print(ask_teacher_question("Hi! My name is Leo and I'm 7 years old.", chat_history))

# Turn 2 (Testing if it remembers 'Leo' and '7 years old')
print("\n--- TEST 2 ---")
print(ask_teacher_question("What is my name and what is a good math problem for me?", chat_history))

--- TEST 1 ---
Hi Leo! Nice to meet you! I'm here to help you learn new things. Think of me like a friendly guide on a learning adventure. 🧭

Since you're 7, you're probably in **first or second grade**. That's awesome! You're learning to read bigger books, add and subtract, and discover how our world works.

Here's how I'll help you:
1. **I'll explain things simply** – like how you'd explain a game to a friend.
2. **I'll use fun examples** – like comparing math to sharing cookies!
3. **I'll give you steps** – so things aren't confusing.

**For example**, if you asked me how plants drink water:
1. Plants have tiny "straws" in their stems called xylem (zy-lem).
2. They suck up water from the soil like a paper towel soaks up a spill.
3. The water travels to the leaves to help make food from sunlight!
**Final answer:** Plants drink through their roots and stems like using straws.

**Want to try a practice question?**
1. If you have 3 toy cars and get 2 more, how many do you have?
2. What 